In [0]:
from pyspark.sql.functions import col, sum, when, regexp_replace
import os
from dotenv import load_dotenv
load_dotenv()

In [0]:
def null_counts(df):
    exprs = []
    for c in df.columns:
        condition = col(c).isNull()

        null_expr = sum(when(condition, 1).otherwise(0))

        exprs.append(null_expr.alias(c))

    return df.select(exprs)

In [0]:
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

## Customer 


In [0]:
customer_df=spark.sql(f"""select * from {BRONZE_SCHEMA_PATH}.`bronze_customer`""")
null_counts(customer_df).show()

In [0]:
customer_df.select("industry_type").distinct().show()

In [0]:
customer_df.select("country").distinct().show()

## Employee

In [0]:
employee_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_employee`""")
display(null_counts(employee_df))

In [0]:
employee_df.select("region").distinct().show()

In [0]:
employee_df.select("role").distinct().show()

## Products

In [0]:
product_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_product`""")

In [0]:
display(null_counts(product_df))

In [0]:
product_df.select("billing_cycle").distinct().show()

In [0]:
product_df.select("plan_name").distinct().show()

## Opportunity Table

In [0]:
opportunity_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_opportunity`""")

In [0]:
display(null_counts(opportunity_df))

In [0]:
opportunity_df.select("close_status").distinct().show()
opportunity_df.select("contract_term").distinct().show()

In [0]:
display(opportunity_df.filter(col("revenue_amount").rlike(r"[^0-9]")))

In [0]:
opportunity_df=opportunity_df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), r"[£$€,]", "").cast("double")
)

In [0]:
display(opportunity_df)